# How does a Bike-share navigate speedy success?

## Overview

This notebook demonstrates end-to-end data analysis workflow focusing on:
- Data preparation and quality assessment
- Exploratory analysis and pattern discovery
- Visualization of key insights
- Actionable recommendations

**Key question:** What distinguishes casual riders from annual members, and how can we convert more casual riders into memberships?

**Data source:** [Public Divvy trip data](https://divvy-tripdata.s3.amazonaws.com/index.html) as a realistic proxy for Cyclistic.

In [1]:
"""Demonstrates a complete data analysis pipeline."""

import pandas as pd
from pathlib import Path

## Load data

Load up to the most recent 12 months of trip data, stored as individual CSV files in `data/raw/`.

In [2]:
# Get list of raw data files sorted by date (up to 12 most recent months)
base = Path('../data/raw')
files = sorted(base.glob("*-divvy-tripdata/*.csv"))[-12:]

# Load files as a list of dataframes
dfs = [pd.read_csv(file) for file in files]

# Extract months from file names
months_pretty = [
    pd.to_datetime(f.parent.name[:6], format="%Y%m").strftime("%B %Y")
    for f in files
]

print(f"Loaded {len(dfs)} month(s) data as dataframes.")
print(f"Month(s): {', '.join(months_pretty)}.")

Loaded 12 month(s) data as dataframes.
Month(s): April 2025, May 2025, June 2025, July 2025, August 2025, September 2025, October 2025, November 2025, December 2025, January 2026, February 2026, March 2026.


## Standardize schema

Standardize column names, column types, and `member_casual` labels across dataframes.

In [3]:
# Fix most common inconsistencies (if any)
updated_dfs = []
for df in dfs:
    # Standardize column names
    df = df.rename(columns={
        'trip_id': 'ride_id', 'usertype': 'member_casual',
        'start_time': 'started_at', 'end_time': 'ended_at',
        'from_station_id': 'start_station_id', 'to_station_id': 'end_station_id',
        'from_station_name': 'start_station_name', 'to_station_name': 'end_station_name'
    })

    # Standardize column dtypes
    df['ride_id'] = df['ride_id'].astype(str)
    df['started_at'] = pd.to_datetime(df['started_at'], utc=True)
    df['ended_at'] = pd.to_datetime(df['ended_at'], utc=True)

    # Standardize `member_casual` labels
    df['member_casual'] = df['member_casual'].replace({
        'Subscriber': 'member', 'Customer': 'casual'
    })
    updated_dfs.append(df)
dfs = updated_dfs

# Standardize columns across files
required_columns = {
    'ride_id', 'member_casual', 'started_at', 'ended_at',
    'start_station_id', 'end_station_id',
    'start_station_name', 'end_station_name'
}
for i, df in enumerate(dfs):
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"File {files[i]} missing columns: {missing_columns}")

print("Standardized column names, column types, and member/casual labels across dataframes.")

Standardized column names, column types, and member/casual labels across dataframes.


## Combine datasets

Bind required rows from multiple dataframes into single dataframe and remove exact duplicates.

In [4]:
# Display floats with commas and up to two decimals (no scientific notation)
pd.options.display.float_format = "{:,.2f}".format

all_trips = pd.concat(
    [df[list(required_columns)] for df in dfs],
    ignore_index=True
)
before = len(all_trips)
all_trips = all_trips.drop_duplicates()
print("Combined required columns from multiple dataframes into a single dataframe.")
print(f"Deduplicated {before - len(all_trips)} rows.")
print(f"Total trips in combined dataframe: {len(all_trips):,}.")

Combined required columns from multiple dataframes into a single dataframe.
Deduplicated 0 rows.
Total trips in combined dataframe: 5,620,544.


## Display summary statistics

Display summary statistics like row count, columns, data types, and missing values.

In [5]:
def view_stat_summary(df: pd.DataFrame) -> None:
    """Print a quick summary of a dataframe's shape, dtypes, and missing values."""
    print("=== Dataset Overview ===")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns.")

    print("\n=== Data Types ===")
    print(df.dtypes)

    print("\n=== Missing Values ===")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0].map(lambda x: f"{x:,}"))
    else:
        print("No missing values detected.")

view_stat_summary(all_trips)

=== Dataset Overview ===
Shape: 5,620,544 rows x 8 columns.

=== Data Types ===
start_station_id                      str
end_station_id                        str
member_casual                         str
started_at            datetime64[us, UTC]
ride_id                               str
end_station_name                      str
ended_at              datetime64[us, UTC]
start_station_name                    str
dtype: object

=== Missing Values ===
start_station_id      1,194,952
end_station_id        1,259,214
end_station_name      1,259,214
start_station_name    1,194,952
dtype: str


## Address missing values

Remove rows with missing values in important columns like `ride_id`, `end_station_id`, and `start_station_id` for data quality.

In [6]:
# Drop rows with missing values
before = len(all_trips)
all_trips = all_trips.dropna(subset=['ride_id', 'end_station_id', 'start_station_id'])
print(f"Removed {before - len(all_trips):,} rows with missing values in important columns for data quality.")

Removed 1,881,299 rows with missing values in important columns for data quality.


## Derive new features

- Compute ride duration in seconds.
- Compute time features like month and day.

In [7]:
# Compute ride duration
all_trips['ride_duration'] = (all_trips['ended_at'] - all_trips['started_at']).dt.total_seconds()

# Derive time features
all_trips['started_month'] = all_trips['started_at'].dt.month_name().str[:3]
all_trips['started_day'] = all_trips['started_at'].dt.day_name().str[:3]
print("Derived new features: ride duration in seconds, time features like month and day.")

# Convert time features to ordered categorical for consistent ordering
for col, col_order in {
    'started_month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    'started_day': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
}.items():
    pd.Categorical(all_trips[col], categories=col_order, ordered=True)
print("Converted time features to ordered categorical for consistent ordering.")

print("\n=== Basic Statistics ===")
print(df.describe(include='all'))

Derived new features: ride duration in seconds, time features like month and day.
Converted time features to ordered categorical for consistent ordering.

=== Basic Statistics ===
                 ride_id  rideable_type                        started_at  \
count             317037         317037                            317037   
unique            317037              2                               NaN   
top     AE72B07EAC06D106  electric_bike                               NaN   
freq                   1         220935                               NaN   
mean                 NaN            NaN  2026-03-18 03:43:13.530539+00:00   
min                  NaN            NaN  2026-02-28 10:48:22.074000+00:00   
25%                  NaN            NaN  2026-03-09 18:08:10.660000+00:00   
50%                  NaN            NaN  2026-03-19 17:18:47.999000+00:00   
75%                  NaN            NaN  2026-03-25 19:25:30.676000+00:00   
max                  NaN            NaN  2026-03-3

## Remove irrelevant trips

- Remove trips with negative or zero ride durations.
- Remove top 1% ride durations as extreme outliers.
- Remove trips that started out of expected date bounds.

In [8]:
# Drop rows with negative or zero ride durations
before = len(all_trips)
all_trips = all_trips[all_trips['ride_duration'] > 0].copy()
print(f"Removed {before - len(all_trips):,} trips with negative or zero ride durations.")

# Filter out top 1% durations to limit extreme outliers
before = len(all_trips)
duration_threshold = all_trips['ride_duration'].quantile(0.99)
all_trips = all_trips[
  all_trips['ride_duration'] < duration_threshold
].copy()
print(f"Removed {before - len(all_trips):,} top 1% ride durations as extreme outliers.")

# Drop rows out of expected date bounds from loaded months
before = len(all_trips)
min_date = pd.to_datetime(files[0].parent.name[:6], format="%Y%m", utc=True)
max_date = (
    pd.to_datetime(files[-1].parent.name[:6], format="%Y%m", utc=True)
    + pd.offsets.MonthEnd()
    + pd.Timedelta(days=1)
)
all_trips = all_trips[(all_trips['started_at'] >= min_date) &
                      (all_trips['started_at'] < max_date)]
print(f"Removed {before - len(all_trips):,} trips that started out of expected date bounds.")

print(f"Total trips after cleaning: {len(all_trips):,}.")

Removed 19 trips with negative or zero ride durations.
Removed 37,393 top 1% ride durations as extreme outliers.
Removed 11 trips that started out of expected date bounds.
Total trips after cleaning: 3,701,822.


## Statistical summary post-cleaning

Review row counts, columns, data types, and missing values after data cleaning and standardization.

In [9]:
view_stat_summary(all_trips)
print("\n=== Basic Statistics ===")
print(all_trips.describe(include='all'))

=== Dataset Overview ===
Shape: 3,701,822 rows x 11 columns.

=== Data Types ===
start_station_id                      str
end_station_id                        str
member_casual                         str
started_at            datetime64[us, UTC]
ride_id                               str
end_station_name                      str
ended_at              datetime64[us, UTC]
start_station_name                    str
ride_duration                     float64
started_month                         str
started_day                           str
dtype: object

=== Missing Values ===
No missing values detected.

=== Basic Statistics ===
       start_station_id end_station_id member_casual  \
count           3701822        3701822       3701822   
unique             3203           3244             2   
top            CHI01747       CHI01747        member   
freq              50954          50979       2401191   
mean                NaN            NaN           NaN   
min                 NaN      